In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=8hu5ImeRnldQNQRKQ9vDVKeKpVJi9R&access_type=offline&code_challenge=ZGwjtg7ZypX9XP1nugM4WvLIoN6OnwmVMLYVXdqzpBY&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"


def gcs_conf(credentials_path=None, project="open-targets-genetics-dev") -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize": "2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning("To change the config restart the session and use the `extended_spark_conf` parameter.")
        return self.spark.sparkContext.getConf().getAll()


session = GentropySession()

Loading BokehJS ...

/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/13 11:38:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
path_to_release_folder = "gs://open-targets-data-releases/25.06/"


si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study/")
sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set/")

sl_eff = session.spark.read.parquet(
    "gs://genetics-portal-dev-analysis/ss60/gentropy-manuscript/chapters/variant-effect-prediction/25.07/lead_variant_effect"
)

l2g_full = session.spark.read.parquet(
    "gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_of_prioritised_genes_per_CS.parquet"
)

In [37]:
training_set = session.spark.read.json(
    "gs://genetics-portal-dev-analysis/yt4/2506_release/training_set/20250625_gentropy_paper_v1.json"
)

In [38]:
test = session.spark.read.parquet("./data/test_v3.parquet")
test.count()

18611

In [39]:
test = test.select("studyLocusId", "geneId", "goldStandardSet").withColumn(
    "goldStandardSet",
    f.when(test["goldStandardSet"] == 1, "positive")
    .when(test["goldStandardSet"] == 0, "negative")
    .otherwise(test["goldStandardSet"]),
)
test.show(1)

+--------------------+---------------+---------------+
|        studyLocusId|         geneId|goldStandardSet|
+--------------------+---------------+---------------+
|02a6e01e6b6597ac7...|ENSG00000105397|       positive|
+--------------------+---------------+---------------+
only showing top 1 row



In [40]:
target = session.spark.read.csv("./data/target_with_constraints_2509.csv", header=True, inferSchema=True)
target.count()

20083

In [41]:
training_set.show(1)

+--------------------+---------------+---------------+------------+--------------------+---------------+
|          diseaseIds|         geneId|goldStandardSet|     studyId|        studyLocusId|      variantId|
+--------------------+---------------+---------------+------------+--------------------+---------------+
|[EFO_0004611, EFO...|ENSG00000130173|       negative|GCST90091598|08ef835a25f0bf2c8...|19_11079858_G_A|
+--------------------+---------------+---------------+------------+--------------------+---------------+
only showing top 1 row



In [42]:
target.show()

+---------------+--------------+----------+----------+----------+
|       targetId|       biotype|syn_constr|mis_constr|lof_constr|
+---------------+--------------+----------+----------+----------+
|ENSG00000000003|protein_coding|      NULL|      NULL|      NULL|
|ENSG00000000005|protein_coding|      NULL|      NULL|      NULL|
|ENSG00000001084|protein_coding|  -0.18886|    2.1807|    -0.404|
|ENSG00000003137|protein_coding|  -0.23942|    1.2829|    -0.371|
|ENSG00000004059|protein_coding|   -0.6046|    2.2684|    -0.685|
|ENSG00000004777|protein_coding|   0.51337|    1.7937|     -0.59|
|ENSG00000004799|protein_coding|  -0.22863|  0.049131|    -1.244|
|ENSG00000004809|protein_coding|  -0.38221|  0.036819|    -1.272|
|ENSG00000004866|protein_coding|-0.0071719|    3.6992|     -0.52|
|ENSG00000005075|protein_coding|   0.77255|    1.5237|    -1.886|
|ENSG00000005436|protein_coding|   -2.1564|   -2.8594|    -1.945|
|ENSG00000005700|protein_coding|   0.77043|     1.881|    -0.626|
|ENSG00000

In [43]:
target = target.withColumnRenamed("targetId", "geneId")

In [44]:
test = test.join(target, on="geneId", how="inner").cache()
test.count()

25/11/13 12:06:49 WARN CacheManager: Asked to cache already cached data.


18611

In [45]:
training_set.count()

132970

In [46]:
training_set = training_set.join(target, on="geneId", how="inner").cache()
training_set.count()

25/11/13 12:06:51 WARN CacheManager: Asked to cache already cached data.


132970

In [47]:
test.show()

+---------------+--------------------+---------------+--------------+----------+----------+----------+
|         geneId|        studyLocusId|goldStandardSet|       biotype|syn_constr|mis_constr|lof_constr|
+---------------+--------------------+---------------+--------------+----------+----------+----------+
|ENSG00000004059|1e374979beade123a...|       negative|protein_coding|   -0.6046|    2.2684|    -0.685|
|ENSG00000004059|202fb7a521a317379...|       negative|protein_coding|   -0.6046|    2.2684|    -0.685|
|ENSG00000004059|b6d69426f1ae617de...|       negative|protein_coding|   -0.6046|    2.2684|    -0.685|
|ENSG00000004059|bef4073281dc376eb...|       negative|protein_coding|   -0.6046|    2.2684|    -0.685|
|ENSG00000004059|c3836ea4b33c5b374...|       negative|protein_coding|   -0.6046|    2.2684|    -0.685|
|ENSG00000004059|951fac749ab653143...|       negative|protein_coding|   -0.6046|    2.2684|    -0.685|
|ENSG00000004059|23fa7b74711ecf15e...|       negative|protein_coding|   -

In [25]:
test.select("mis_constr", "lof_constr").describe().show()

+-------+------------------+-------------------+
|summary|        mis_constr|         lof_constr|
+-------+------------------+-------------------+
|  count|             18539|              18157|
|   mean|1.1648516550999937|-0.9215826953791968|
| stddev|1.5029327764146094|0.43499080216329955|
|    min|           -3.5405|             -1.976|
|    max|             8.811|             -0.102|
+-------+------------------+-------------------+



In [48]:
from pyspark.sql import Window

# Window ordered by lof_constr descending
w = Window.partitionBy("studyLocusId").orderBy(f.desc("lof_constr"))

# Option A: pick a single highest (row_number -> exactly one marked 1 per locus)
training_set = (
    training_set.withColumn("row_nr_lof", f.row_number().over(w))
    .withColumn("is_top_lof", f.when(f.col("row_nr_lof") == 1, f.lit(1)).otherwise(f.lit(0)))
    .drop("row_nr_lof")
)

In [49]:
training_set.show(10)

+---------------+-------------+---------------+------------+--------------------+---------------+--------------+----------+----------+----------+----------+
|         geneId|   diseaseIds|goldStandardSet|     studyId|        studyLocusId|      variantId|       biotype|syn_constr|mis_constr|lof_constr|is_top_lof|
+---------------+-------------+---------------+------------+--------------------+---------------+--------------+----------+----------+----------+----------+
|ENSG00000159579|[EFO_0004612]|       negative|GCST90018956|005bc8624f8dd7f7c...|16_56973441_C_T|protein_coding|    1.3958|    3.0301|    -0.445|         1|
|ENSG00000006210|[EFO_0004612]|       negative|GCST90018956|005bc8624f8dd7f7c...|16_56973441_C_T|protein_coding|   -0.9394|   0.43367|    -0.577|         0|
|ENSG00000140853|[EFO_0004612]|       negative|GCST90018956|005bc8624f8dd7f7c...|16_56973441_C_T|protein_coding|   0.51987|      2.59|    -0.651|         0|
|ENSG00000051108|[EFO_0004612]|       negative|GCST9001895

In [50]:
chemblDrugEnrichment.studyLocusId_based_evidence_table_vs_training_set(
    table_with_score=training_set,
    training_set=training_set,
    score_column="is_top_lof",
    min_score=0.5,
    name_of_the_evidence="l2g",
)

,Evidence,TP,TN,FP,FN,Sensitivity (recall),Specificity (selectivity),PPV (precision),FDR,Balanced_accuracy
0,l2g,911,117126,7324,7609,0.106925,0.941149,0.110625,0.889375,0.524037


In [ ]:
# ...existing code...
from scipy.stats import fisher_exact, chi2_contingency
from pyspark.sql import functions as f

# robustly normalise goldStandardSet to string and check common truthy values
training_set = training_set.withColumn(
    "gold_binary",
    f.when(
        f.lower(f.col("goldStandardSet").cast("string")).isin("positive", "1", "true", "t", "yes"),
        f.lit(1),
    ).otherwise(f.lit(0)),
)

# compute contingency counts
a = int(training_set.filter((f.col("is_top_lof") == 1) & (f.col("gold_binary") == 1)).count())
b = int(training_set.filter((f.col("is_top_lof") == 1) & (f.col("gold_binary") == 0)).count())
c = int(training_set.filter((f.col("is_top_lof") == 0) & (f.col("gold_binary") == 1)).count())
d = int(training_set.filter((f.col("is_top_lof") == 0) & (f.col("gold_binary") == 0)).count())

contingency = [[a, b], [c, d]]
print("Contingency table (rows=is_top_lof 1/0, cols=gold 1/0):", contingency)

oddsratio, p_value = fisher_exact(contingency, alternative="two-sided")
print(f"Fisher exact OR = {oddsratio}, p-value = {p_value}")

chi2, chi_p, dof, expected = chi2_contingency(contingency)
print(f"Chi-square p-value = {chi_p}, chi2 = {chi2}, dof = {dof}")
# ...existing code...

Contingency table (rows=is_top_lof 1/0, cols=gold 1/0): [[911, 7324], [7609, 117126]]
Fisher exact OR = 1.9146780964994528, p-value = 3.132956516117402e-60
Chi-square p-value = 8.858727746943528e-71, chi2 = 316.3894481597703, dof = 1


25/11/13 17:07:59 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1205877 ms exceeds timeout 120000 ms
25/11/13 17:07:59 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/13 17:08:01 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$